In [13]:
# Initialize Spark
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

Spark Version: 4.1.1


In [2]:
# Download the specific file mentioned in the homework
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

# Read the Parquet file into a DataFrame
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

# Show the schema to understand the data structure
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [3]:
# Repartition to 4 and save
df.repartition(4).write.parquet('data/pq/yellow/2025/11/', mode='overwrite')

# To find the average file size (Question 2), check the folder in your terminal:
# !ls -lh data/pq/yellow/2025/11/

In [4]:
from pyspark.sql import functions as F

# Question 3: Trips on Nov 15th
df_count = df.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter(F.col('pickup_date') == '2025-11-15') \
    .count()

print(f"Trips on Nov 15th: {df_count}")

# Question 4: Longest trip in hours
# Calculate (Dropoff - Pickup) in seconds, then convert to hours
df = df.withColumn('duration_hrs', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
)

max_duration = df.select(F.max('duration_hrs')).collect()[0][0]
print(f"Longest trip: {max_duration} hours")

Trips on Nov 15th: 162604
Longest trip: 90.64666666666666 hours


In [ ]:
# Download the file
# !wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 11:29:38--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.213, 18.239.115.4, 18.239.115.86, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.213|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 11:29:38 (265 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [ ]:
# Load the Yellow Taxi data
df_trips = spark.read.parquet('yellow_tripdata_2025-11.parquet')

# Load the Zone Lookup data
df_zones = spark.read.csv('taxi_zone_lookup.csv', header=True, inferSchema=True)

# Register them as Temporary Views to use Spark SQL
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [7]:
# Using Spark SQL to find the least frequent pickup zone
result = spark.sql("""
    SELECT
        z.Zone,
        COUNT(1) as trip_count
    FROM
        trips t
    JOIN
        zones z ON t.PULocationID = z.LocationID
    GROUP BY
        1
    ORDER BY
        trip_count ASC
    LIMIT 1
""")

result.show()

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
+--------------------+----------+



In [14]:
spark.stop()